In [3]:
import pandas as pd
india = pd.read_csv(r"../data/raw/India.csv")

print("Shape:", india.shape)

print("\nColumns:")
print(india.columns.tolist())

print("\nFirst 5 rows:")
display(india.head())

print("\nData types:")
display(india.dtypes)

print("\nMissing values:")
display(india.isnull().sum())

Shape: (73632, 9)

Columns:
['State', 'ST_CEN_CD', 'District', 'DISTRICT', 'DT_CEN_CD', 'Category', 'Indicator', 'NFHS 5', 'NFHS 4']

First 5 rows:


C:\Users\ASUS\AppData\Local\Temp\ipykernel_13260\4152248674.py:2: DtypeWarning: Columns (0: DT_CEN_CD) have mixed types. Specify dtype option on import or set low_memory=False.
  india = pd.read_csv(r"../data/raw/India.csv")


,State,ST_CEN_CD,District,DISTRICT,DT_CEN_CD,Category,Indicator,NFHS 5,NFHS 4
0,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Population and Household Profile,Female population age 6 years and above who ev...,78.0,77.2
1,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Population and Household Profile,Population below age 15 years (%),23.0,25.4
2,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Population and Household Profile,Sex ratio of the total population (females per...,973.0,957.0
3,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Population and Household Profile,Sex ratio at birth for children born in the la...,927.0,1060.0
4,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Population and Household Profile,Children under age 5 years whose birth was reg...,98.0,99.3



Data types:


State            str
ST_CEN_CD      int64
District         str
DISTRICT         str
DT_CEN_CD     object
Category         str
Indicator        str
NFHS 5       float64
NFHS 4       float64
dtype: object


Missing values:


State           0
ST_CEN_CD       0
District        0
DISTRICT     6656
DT_CEN_CD    6760
Category        0
Indicator       0
NFHS 5          0
NFHS 4          0
dtype: int64

In [4]:
# Check combinations of missing district identifiers

print("DISTRICT and DT_CEN_CD missingness:")

display(
    pd.crosstab(
        india['DISTRICT'].isna(),
        india['DT_CEN_CD'].isna(),
        margins=True
    )
)

DISTRICT and DT_CEN_CD missingness:


DT_CEN_CD,False,True,All
DISTRICT,,,
False,66767,209,66976
True,105,6551,6656
All,66872,6760,73632


In [5]:
missing_summary = pd.DataFrame({
    'Identifier combination': [
        'Both present',
        'District present, code missing',
        'District missing, code present',
        'Both missing'
    ],
    'Records': [
        66767,
        209,
        105,
        6551
    ]
})

missing_summary['Percentage'] = (
    missing_summary['Records'] / len(india) * 100
)

display(missing_summary.round(2))

,Identifier combination,Records,Percentage
0,Both present,66767,90.68
1,"District present, code missing",209,0.28
2,"District missing, code present",105,0.14
3,Both missing,6551,8.90


In [6]:
both_missing = india[
    india['DISTRICT'].isna() &
    india['DT_CEN_CD'].isna()
].copy()

print("Total:", len(both_missing))

display(both_missing.head(20))

Total: 6551


,State,ST_CEN_CD,District,DISTRICT,DT_CEN_CD,Category,Indicator,NFHS 5,NFHS 4
2184,Arunachal Pradesh,12,Kra Daadi,NaN,NaN,Population and Household Profile,Female population age 6 years and above who ev...,58.1,0.0
2185,Arunachal Pradesh,12,Kra Daadi,NaN,NaN,Population and Household Profile,Population below age 15 years (%),33.5,0.0
2186,Arunachal Pradesh,12,Kra Daadi,NaN,NaN,Population and Household Profile,Sex ratio of the total population (females per...,976.0,0.0
2187,Arunachal Pradesh,12,Kra Daadi,NaN,NaN,Population and Household Profile,Sex ratio at birth for children born in the la...,867.0,0.0
2188,Arunachal Pradesh,12,Kra Daadi,NaN,NaN,Population and Household Profile,Children under age 5 years whose birth was reg...,92.7,0.0
2189,Arunachal Pradesh,12,Kra Daadi,NaN,NaN,Population and Household Profile,Deaths in the last 3 years registered with the...,30.7,0.0
2190,Arunachal Pradesh,12,Kra Daadi,NaN,NaN,Population and Household Profile,Population living in households with electrici...,99.7,0.0
2191,Arunachal Pradesh,12,Kra Daadi,NaN,NaN,Population and Household Profile,Population living in households with an improv...,99.8,0.0
2192,Arunachal Pradesh,12,Kra Daadi,NaN,NaN,Population and Household Profile,Population living in households that use an im...,89.3,0.0
2193,Arunachal Pradesh,12,Kra Daadi,NaN,NaN,Population and Household Profile,Households using clean fuel for cooking (%),78.3,0.0


In [7]:
print("Unique states:", both_missing['State'].nunique())

print("Unique district names:",
      both_missing['District'].nunique())

print("Missing District names:",
      both_missing['District'].isna().sum())

print("Missing NFHS-4 values:",
      both_missing['NFHS 4'].isna().sum())

print("NFHS-4 values equal to zero:",
      (both_missing['NFHS 4'] == 0).sum())

Unique states: 14
Unique district names: 64
Missing District names: 0
Missing NFHS-4 values: 0
NFHS-4 values equal to zero: 6551


In [8]:
districts_without_codes = (
    both_missing.groupby('State')
    .agg(
        number_of_districts=('District', 'nunique'),
        number_of_records=('Indicator', 'size')
    )
    .sort_values('number_of_districts', ascending=False)
    .reset_index()
)

display(districts_without_codes)

,State,number_of_districts,number_of_records
0,Telangana,19,1976
1,Chhattisgarh,9,935
2,Gujarat,7,728
3,Assam,6,624
4,Meghalaya,4,311
5,Arunachal Pradesh,4,416
6,Tripura,4,416
7,Uttar Pradesh,4,416
8,NCT of Delhi,2,208
9,Punjab,2,208


In [9]:
for state in sorted(both_missing['State'].unique()):
    print(repr(state), len(state))

'Arunachal Pradesh' 17
'Assam' 5
'Chhattisgarh' 12
'Chhattisgarh ' 13
'Gujarat' 7
'Haryana' 7
'Madhya Pradesh' 14
'Maharashtra' 11
'Meghalaya' 9
'NCT of Delhi' 12
'Punjab' 6
'Telangana' 9
'Tripura' 7
'Uttar Pradesh' 13


In [10]:
india_analysis = india.copy()

india_analysis['State_clean'] = (
    india_analysis['State'].str.strip()
)

print("Original unique states:",
      india_analysis['State'].nunique())

print("Cleaned unique states:",
      india_analysis['State_clean'].nunique())

print("\nCleaned state names:")
print(sorted(india_analysis['State_clean'].unique()))

Original unique states: 38
Cleaned unique states: 37

Cleaned state names:
['Andaman & Nicobar Island', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra & Nagar Haveli', 'Daman & Diu', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu & Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'NCT of Delhi', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [11]:
both_missing_clean = india_analysis[
    india_analysis['DISTRICT'].isna() &
    india_analysis['DT_CEN_CD'].isna()
].copy()

district_distribution = (
    both_missing_clean.groupby('State_clean')
    .agg(
        number_of_districts=('District', 'nunique'),
        number_of_records=('Indicator', 'size')
    )
    .reset_index()
    .sort_values('number_of_records', ascending=False)
)

display(district_distribution)

,State_clean,number_of_districts,number_of_records
10,Telangana,19,1976
2,Chhattisgarh,9,936
3,Gujarat,7,728
1,Assam,6,624
0,Arunachal Pradesh,4,416
12,Uttar Pradesh,4,416
11,Tripura,4,416
7,Meghalaya,4,311
8,NCT of Delhi,2,208
9,Punjab,2,208


In [12]:
district_indicator_counts = (
    both_missing_clean.groupby(
        ['State_clean', 'District']
    )
    .agg(
        total_records=('Indicator', 'size'),
        unique_indicators=('Indicator', 'nunique')
    )
    .reset_index()
    .sort_values(
        ['State_clean', 'total_records']
    )
)

display(district_indicator_counts.to_string(index=False))

'      State_clean                 District  total_records  unique_indicators\nArunachal Pradesh               Kra Daadi             104                104\nArunachal Pradesh                Longding             104                104\nArunachal Pradesh                  Namsai             104                104\nArunachal Pradesh                   Siang             104                104\n            Assam                Biswanath            104                104\n            Assam                Charaideo            104                104\n            Assam                    Hojai            104                104\n            Assam                   Majuli            104                104\n            Assam  South Salmara Mancachar            104                104\n            Assam       West Karbi Anglong            104                104\n     Chhattisgarh                    Sukma            103                103\n     Chhattisgarh                    Balod            104      

In [13]:
# Most common indicator set in the subset
reference_indicators = set(
    both_missing_clean['Indicator'].value_counts().head(104).index
)

# Inspect the five irregular districts
irregular_names = [
    'Sukma',
    'Surajpur',
    'North Garo Hills',
    'South West Garo Hills',
    'West Jaintia Hills'
]

for district in irregular_names:

    district_data = both_missing_clean[
        both_missing_clean['District'] == district
    ]

    present = set(district_data['Indicator'])

    missing = reference_indicators - present
    additional = present - reference_indicators

    print(f"\nDistrict: {district}")
    print("Records:", len(district_data))
    print("Unique indicators:", len(present))
    print("Missing indicators:", missing)
    print("Additional indicators:", additional)


District: Sukma
Records: 103
Unique indicators: 103
Missing indicators: {'Men age 15 years and above who consume alcohol (%)'}
Additional indicators: set()

District: Surajpur
Records: 105
Unique indicators: 104
Missing indicators: set()
Additional indicators: set()

District: North Garo Hills
Records: 103
Unique indicators: 103
Missing indicators: {'All women age 15-19 years who are anaemic (%)'}
Additional indicators: set()

District: South West Garo Hills
Records: 103
Unique indicators: 103
Missing indicators: {'All women age 15-19 years who are anaemic (%)'}
Additional indicators: set()

District: West Jaintia Hills
Records: 1
Unique indicators: 1
Missing indicators: {'Sex ratio at birth for children born in the last five years (females per 1,000 males)', 'Children age 12-23 months who have received 3 doses of rotavirus vaccine (%)', 'Population living in households with an improved drinkingwater source (%)', 'Children age 12-23 months who have received 3 doses of penta or hepatit

In [14]:
surajpur_data = both_missing_clean[
    both_missing_clean['District'] == 'Surajpur'
].copy()

surajpur_duplicates = surajpur_data[
    surajpur_data.duplicated(
        subset=['Indicator'],
        keep=False
    )
].sort_values('Indicator')

display(
    surajpur_duplicates[
        ['State', 'District', 'Category',
         'Indicator', 'NFHS 5', 'NFHS 4']
    ]
)

,State,District,Category,Indicator,NFHS 5,NFHS 4
13727,Chhattisgarh,Surajpur,Tobacco Use and Alcohol Consumption among Adults,Men age 15 years and above who consume alcohol...,54.2,0.0
13831,Chhattisgarh,Surajpur,Tobacco Use and Alcohol Consumption among Adults,Men age 15 years and above who consume alcohol...,43.4,0.0


In [15]:
display(
    india.loc[[13727, 13831]].T
)

,13727,13831
State,Chhattisgarh,Chhattisgarh
ST_CEN_CD,22,22
District,Surajpur,Surajpur
DISTRICT,NaN,NaN
DT_CEN_CD,NaN,NaN
Category,Tobacco Use and Alcohol Consumption among Adults,Tobacco Use and Alcohol Consumption among Adults
Indicator,Men age 15 years and above who consume alcohol...,Men age 15 years and above who consume alcohol...
NFHS 5,54.2,43.4
NFHS 4,0.0,0.0


In [16]:
surajpur_indicator = (
    'Men age 15 years and above who consume alcohol (%)'
)

duplicate_indicator = india_analysis[
    india_analysis['Indicator'].str.strip()
    == surajpur_indicator
]

display(
    duplicate_indicator[
        [
            'State_clean',
            'District',
            'DISTRICT',
            'DT_CEN_CD',
            'NFHS 5',
            'NFHS 4'
        ]
    ]
)

,State_clean,District,DISTRICT,DT_CEN_CD,NFHS 5,NFHS 4
103,Andaman & Nicobar Island,Nicobar,Nicobar_1,1.0,64.5,0.0
207,Andaman & Nicobar Island,North & Middle Andaman,North & Middle Andaman_2,2.0,45.3,0.0
311,Andaman & Nicobar Island,South Andaman,South Andaman_3,3.0,32.8,0.0
415,Andhra Pradesh,Anantapur,Anantapur_12,12.0,16.8,0.0
519,Andhra Pradesh,Chittoor,Chittoor_13,13.0,18.3,0.0
...,...,...,...,...,...,...
73215,West Bengal,Purba Barddhaman,Pashchim Medinipur_18,18,22.9,0.0
73319,West Bengal,Purba Medinipur,Purba Medinipur_19,19,14.8,0.0
73423,West Bengal,Puruliya,Puruliya_14,14,17.8,0.0
73527,West Bengal,South Twenty Four Parganas,South 24 Parganas_17,17,16.5,0.0


In [17]:
print("Total records:", len(duplicate_indicator))

print("\nNFHS-4 value distribution:")
display(
    duplicate_indicator['NFHS 4']
    .value_counts(dropna=False)
)

print("\nUnique NFHS-4 values:")
print(duplicate_indicator['NFHS 4'].unique())

Total records: 708

NFHS-4 value distribution:


NFHS 4
0.0    708
Name: count, dtype: int64


Unique NFHS-4 values:
[0.]


In [18]:
zero_audit = (
    india.groupby('Indicator')
    .agg(
        total_records=('NFHS 4', 'size'),
        zero_records=('NFHS 4', lambda x: (x == 0).sum()),
        unique_values=('NFHS 4', 'nunique')
    )
    .reset_index()
)

zero_audit['zero_percentage'] = (
    zero_audit['zero_records'] /
    zero_audit['total_records'] * 100
)

display(
    zero_audit.sort_values(
        'zero_percentage',
        ascending=False
    ).head(25)
)

,Indicator,total_records,zero_records,unique_values,zero_percentage
16,Children age 12-23 months who have received 3 ...,708,708,1,100.0
35,Children who received postnatal care from a do...,708,708,1,100.0
21,Children age 24-35 months who have received a ...,708,708,1,100.0
22,Children age 5 years who attended preprimary s...,708,708,1,100.0
47,Female Blood sugar level high or very high (>...,708,708,1,100.0
45,Ever undergone an oral cavity examination for ...,708,708,1,100.0
44,Ever undergone a screening test for cervical c...,708,708,1,100.0
43,Ever undergone a breast examination for breast...,708,708,1,100.0
42,Deaths in the last 3 years registered with the...,708,708,1,100.0
63,Male Blood sugar level high (141-160 mg/dl) (%),1,1,1,100.0


In [19]:
print("Total unique indicators:", len(zero_audit))

print(
    "Indicators with 100% zero values:",
    (zero_audit['zero_percentage'] == 100).sum()
)

print(
    "Indicators with at least 90% zero values:",
    (zero_audit['zero_percentage'] >= 90).sum()
)

print(
    "Indicators with no zero values:",
    (zero_audit['zero_records'] == 0).sum()
)

print("\nDistribution of zero percentages:")
display(
    zero_audit['zero_percentage']
    .describe()
)

Total unique indicators: 105
Indicators with 100% zero values: 27
Indicators with at least 90% zero values: 27
Indicators with no zero values: 0

Distribution of zero percentages:


count    105.000000
mean      44.346247
std       35.166767
min       18.785311
25%       18.785311
50%       19.491525
75%      100.000000
max      100.000000
Name: zero_percentage, dtype: float64

In [20]:
malnutrition_indicators = india[
    india['Indicator'].str.contains(
        'stunted|wasted|underweight',
        case=False,
        na=False
    )
].copy()

malnutrition_audit = (
    malnutrition_indicators.groupby('Indicator')
    .agg(
        total_records=('NFHS 4', 'size'),
        nfhs4_zero=('NFHS 4', lambda x: (x == 0).sum()),
        nfhs5_zero=('NFHS 5', lambda x: (x == 0).sum()),
        nfhs4_unique=('NFHS 4', 'nunique'),
        nfhs5_unique=('NFHS 5', 'nunique')
    )
    .reset_index()
)

display(malnutrition_audit)

,Indicator,total_records,nfhs4_zero,nfhs5_zero,nfhs4_unique,nfhs5_unique
0,Children under 5 years who are severely wasted...,708,134,1,161,162
1,Children under 5 years who are stunted (height...,708,133,1,306,305
2,Children under 5 years who are underweight (we...,708,133,1,331,325
3,Children under 5 years who are wasted (weight ...,708,133,1,262,240


In [21]:
# Extract the three malnutrition indicators

target_indicators = [
    'Children under 5 years who are stunted (height for age) (%)',
    'Children under 5 years who are wasted (weight for height) (%)',
    'Children under 5 years who are underweight (weight for age) (%)'
]

nutrition_data = india_analysis[
    india_analysis['Indicator'].str.strip().isin(target_indicators)
].copy()

print("Shape:", nutrition_data.shape)

display(
    nutrition_data[
        ['State_clean', 'District', 'Indicator', 'NFHS 5', 'NFHS 4']
    ].head(10)
)

Shape: (2124, 10)


,State_clean,District,Indicator,NFHS 5,NFHS 4
72,Andaman & Nicobar Island,Nicobar,Children under 5 years who are stunted (height...,21.6,20.3
73,Andaman & Nicobar Island,Nicobar,Children under 5 years who are wasted (weight ...,15.7,7.4
75,Andaman & Nicobar Island,Nicobar,Children under 5 years who are underweight (we...,24.6,10.5
176,Andaman & Nicobar Island,North & Middle Andaman,Children under 5 years who are stunted (height...,27.0,32.5
177,Andaman & Nicobar Island,North & Middle Andaman,Children under 5 years who are wasted (weight ...,27.0,31.7
179,Andaman & Nicobar Island,North & Middle Andaman,Children under 5 years who are underweight (we...,42.8,33.5
280,Andaman & Nicobar Island,South Andaman,Children under 5 years who are stunted (height...,21.1,20.1
281,Andaman & Nicobar Island,South Andaman,Children under 5 years who are wasted (weight ...,12.6,15.6
283,Andaman & Nicobar Island,South Andaman,Children under 5 years who are underweight (we...,17.4,18.5
384,Andhra Pradesh,Anantapur,Children under 5 years who are stunted (height...,36.0,40.3


In [22]:
nutrition_duplicates = nutrition_data[
    nutrition_data.duplicated(
        subset=['State_clean', 'District', 'Indicator'],
        keep=False
    )
].sort_values(
    ['State_clean', 'District', 'Indicator']
)

print("Duplicate records:", len(nutrition_duplicates))

display(nutrition_duplicates)

Duplicate records: 6


,State,ST_CEN_CD,District,DISTRICT,DT_CEN_CD,Category,Indicator,NFHS 5,NFHS 4,State_clean
25968,Jharkhand,20,Sahibganj,Sahibganj_7,7.0,Child Feeding Practices and Nutritional Status...,Children under 5 years who are stunted (height...,49.1,50.2,Jharkhand
26072,Jharkhand,20,Sahibganj,Saraikela-kharsawan_24,7.0,Child Feeding Practices and Nutritional Status...,Children under 5 years who are stunted (height...,40.0,45.1,Jharkhand
25971,Jharkhand,20,Sahibganj,Sahibganj_7,7.0,Child Feeding Practices and Nutritional Status...,Children under 5 years who are underweight (we...,44.8,49.7,Jharkhand
26075,Jharkhand,20,Sahibganj,Saraikela-kharsawan_24,7.0,Child Feeding Practices and Nutritional Status...,Children under 5 years who are underweight (we...,48.7,52.6,Jharkhand
25969,Jharkhand,20,Sahibganj,Sahibganj_7,7.0,Child Feeding Practices and Nutritional Status...,Children under 5 years who are wasted (weight ...,19.7,24.6,Jharkhand
26073,Jharkhand,20,Sahibganj,Saraikela-kharsawan_24,7.0,Child Feeding Practices and Nutritional Status...,Children under 5 years who are wasted (weight ...,32.9,23.3,Jharkhand


In [23]:
jharkhand_duplicates = india_analysis.loc[
    nutrition_duplicates.index
].copy()

display(
    jharkhand_duplicates[
        [
            'State',
            'ST_CEN_CD',
            'District',
            'DISTRICT',
            'DT_CEN_CD',
            'Category',
            'Indicator',
            'NFHS 5',
            'NFHS 4'
        ]
    ].sort_values(['Indicator', 'DISTRICT'])
)

,State,ST_CEN_CD,District,DISTRICT,DT_CEN_CD,Category,Indicator,NFHS 5,NFHS 4
25968,Jharkhand,20,Sahibganj,Sahibganj_7,7.0,Child Feeding Practices and Nutritional Status...,Children under 5 years who are stunted (height...,49.1,50.2
26072,Jharkhand,20,Sahibganj,Saraikela-kharsawan_24,7.0,Child Feeding Practices and Nutritional Status...,Children under 5 years who are stunted (height...,40.0,45.1
25971,Jharkhand,20,Sahibganj,Sahibganj_7,7.0,Child Feeding Practices and Nutritional Status...,Children under 5 years who are underweight (we...,44.8,49.7
26075,Jharkhand,20,Sahibganj,Saraikela-kharsawan_24,7.0,Child Feeding Practices and Nutritional Status...,Children under 5 years who are underweight (we...,48.7,52.6
25969,Jharkhand,20,Sahibganj,Sahibganj_7,7.0,Child Feeding Practices and Nutritional Status...,Children under 5 years who are wasted (weight ...,19.7,24.6
26073,Jharkhand,20,Sahibganj,Saraikela-kharsawan_24,7.0,Child Feeding Practices and Nutritional Status...,Children under 5 years who are wasted (weight ...,32.9,23.3


In [24]:
jharkhand_geo_check = (
    india_analysis[
        india_analysis['State_clean'] == 'Jharkhand'
    ][
        ['District', 'DISTRICT', 'DT_CEN_CD']
    ]
    .drop_duplicates()
    .sort_values(['District', 'DISTRICT'])
)

display(jharkhand_geo_check.to_string(index=False))

'           District               DISTRICT DT_CEN_CD\n             Bokaro              Bokaro_10      10.0\n             Chatra               Chatra_2       2.0\n            Deoghar              Deoghar_5       5.0\n            Dhanbad              Dhanbad_9       9.0\n              Dumka               Dumka_17      17.0\n             Garhwa               Garhwa_1       1.0\n            Giridih              Giridih_4       4.0\n              Godda                Godda_6       6.0\n              Gumla               Gumla_21      21.0\n         Hazaribagh          Hazaribagh_15      15.0\n            Jamtara             Jamtara_18      18.0\n             Khunti              Khunti_20      20.0\n            Kodarma              Kodarma_3       3.0\n            Latehar             Latehar_14      14.0\n          Lohardaga           Lohardaga_11      11.0\n              Pakur                Pakur_8       8.0\n             Palamu              Palamu_13      13.0\nPashchimi Singhbhum Pashchi

In [25]:
jharkhand_conflict = india_analysis[
    (india_analysis['State_clean'] == 'Jharkhand') &
    (india_analysis['District'] == 'Sahibganj') &
    (india_analysis['DISTRICT'] == 'Saraikela-kharsawan_24')
]

print("Total conflicting records:", len(jharkhand_conflict))
print("Unique indicators:", jharkhand_conflict['Indicator'].nunique())

display(
    jharkhand_conflict[
        [
            'District',
            'DISTRICT',
            'DT_CEN_CD',
            'Indicator',
            'NFHS 5',
            'NFHS 4'
        ]
    ].sort_values('Indicator')
)

Total conflicting records: 104
Unique indicators: 104


,District,DISTRICT,DT_CEN_CD,Indicator,NFHS 5,NFHS 4
26084,Sahibganj,Saraikela-kharsawan_24,7.0,All women age 15-19 years who are anaemic (%),74.9,88.1
26083,Sahibganj,Saraikela-kharsawan_24,7.0,All women age 15-49 years who are anaemic (%),70.2,78.8
26019,Sahibganj,Saraikela-kharsawan_24,7.0,Any method (%),66.9,28.4
26020,Sahibganj,Saraikela-kharsawan_24,7.0,Any modern method (%),52.4,26.6
26038,Sahibganj,Saraikela-kharsawan_24,7.0,Average out-of-pocket expenditure per delivery...,1770.0,1067.0
...,...,...,...,...,...,...
26013,Sahibganj,Saraikela-kharsawan_24,7.0,Women who are literate (%),64.0,0.0
26078,Sahibganj,Saraikela-kharsawan_24,7.0,Women who are overweight or obese,6.4,12.4
26079,Sahibganj,Saraikela-kharsawan_24,7.0,Women who have high risk waisttohip ratio,63.9,0.0
26077,Sahibganj,Saraikela-kharsawan_24,7.0,Women whose Body Mass Index (BMI) is below nor...,20.3,34.8


In [26]:
sahibganj_profiles = india_analysis[
    (india_analysis['State_clean'] == 'Jharkhand') &
    (india_analysis['District'] == 'Sahibganj')
].copy()

profile_comparison = sahibganj_profiles.pivot(
    index='Indicator',
    columns='DISTRICT',
    values='NFHS 5'
)

profile_comparison['difference'] = (
    profile_comparison['Sahibganj_7']
    - profile_comparison['Saraikela-kharsawan_24']
)

display(profile_comparison)

DISTRICT,Sahibganj_7,Saraikela-kharsawan_24,difference
Indicator,,,
All women age 15-19 years who are anaemic (%),67.6,74.9,-7.3
All women age 15-49 years who are anaemic (%),71.5,70.2,1.3
Any method (%),50.4,66.9,-16.5
Any modern method (%),34.0,52.4,-18.4
Average out-of-pocket expenditure per delivery in a public health facility (Rs),1546.0,1770.0,-224.0
...,...,...,...
Women who are literate (%),48.8,64.0,-15.2
Women who are overweight or obese,14.3,6.4,7.9
Women who have high risk waisttohip ratio,66.4,63.9,2.5


In [27]:
profile_identity = india_analysis[
    (india_analysis['State_clean'] == 'Jharkhand') &
    (india_analysis['DISTRICT'].isin([
        'Sahibganj_7',
        'Saraikela-kharsawan_24'
    ]))
].groupby(['DISTRICT', 'District', 'DT_CEN_CD']).agg(
    record_count=('Indicator', 'size'),
    unique_indicators=('Indicator', 'nunique'),
    nfhs5_unique_values=('NFHS 5', 'nunique')
).reset_index()

display(profile_identity)


,DISTRICT,District,DT_CEN_CD,record_count,unique_indicators,nfhs5_unique_values
0,Sahibganj_7,Sahibganj,7.0,104,104,93
1,Saraikela-kharsawan_24,Sahibganj,7.0,104,104,93


In [28]:
display(
    india_analysis[
        india_analysis['DISTRICT'].isin([
            'Sahibganj_7',
            'Saraikela-kharsawan_24'
        ])
    ][
        ['State', 'ST_CEN_CD', 'District', 'DISTRICT', 'DT_CEN_CD']
    ].drop_duplicates()
)

,State,ST_CEN_CD,District,DISTRICT,DT_CEN_CD
25896,Jharkhand,20,Sahibganj,Sahibganj_7,7.0
26000,Jharkhand,20,Sahibganj,Saraikela-kharsawan_24,7.0
